# Turnover, size, price, return, vol — six models, two questions

Independent of the buying-pressure work. **No funds, no holdings, no Barra.** One row is one
(security, quarter); every number comes from price, volume and size.

## Turnover

`close` gives shares outstanding, and from there turnover:

$$\text{shares out}=\frac{\text{market\_cap}}{\text{close}},\qquad
\text{turnover}_{s,q}=\frac{\text{volume}_{s,q}}{\text{shares out}_{s,q}}
=\frac{\text{volume}\times\text{close}}{\text{market\_cap}}$$

`build_panel` prints the median under both readings of `volume` (share volume vs dollar
volume) and picks the plausible one — a quarter's turnover is normally a few tens of percent.
**Check that line**; if neither is plausible the units are not what either formula assumes.

## Features — all known at the close of q

| | |
|---|---|
| `turnover` | volume / shares outstanding, over q |
| `log_mktcap` | log market cap at q |
| `ret_q` | the return **during** q |
| `vol_ret` | dispersion of the last 4 quarterly returns, q included |
| `log_price` | log close at q |

## Targets — both strictly after every feature

| | |
|---|---|
| `turnover_next` | turnover over q+1 |
| `ret_next` | the return over q → q+1 |

Every feature is $\mathcal I_q$-measurable and both targets land after q closes, so a forecast made
at the q close precedes the whole window it is graded on. A quintile sort on a prediction is
tradeable at that close and the spread it earns carries no overlap.

## Models

Five univariate GBMs, one per feature, plus one on all five — and each raw feature scored on
its own as a reference. For a single feature a tree is close to a monotone transform, so
`model:x` and `raw:x` should nearly agree; a large gap means the tree fitted noise.

> **`Q5_Q1_pct` is in next-quarter return for every row**, whatever the target is. That is the
> only way a turnover model and a return model can be read off one scale.


In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())
import numpy as np, pandas as pd
import turnover_study as T
T.check_version()
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)


## 1. Configuration

In [ ]:
CFG = T.Config(
    holdings_path = "manager_holdings/master_batches_return_filtered/master_all_funds_add_filter_ivy_rank_active_rank.parquet",
    inv_type_codes = (401,),
    vol_window   = 4,      # quarters behind vol_ret (current + 3 lags)
    winsorize    = 0.01,   # per-quarter clip; turnover has a long right tail
    min_quarters = 8,      # expanding history, so no survivorship in the universe
    window_q = 28, test_q = 8, step = 8,
)
# If the column is not literally called "close"/"volume", remap it here:
# CFG.col_map["close"] = "close_price"
CFG


## 2. Build the panel

Three lines to read before anything else:

- **`[turnover] using: ...`** — which volume convention was chosen. Wrong here and every
  turnover number below is wrong.
- **`turnover_next available`** — the share of rows with a next quarter. Low means the panel
  is gappy and the turnover target is thin.
- **feature coverage** — a feature at well under 100% is being imputed by the tree, and its
  univariate model is then partly fitting the missingness pattern.

In [ ]:
panel = T.build_panel(CFG)
display(panel[T.FEATURES + T.TARGETS].describe().T.round(4))


In [ ]:
# sanity: turnover should be right-skewed and positive, and the features should not be
# near-duplicates of each other
display(panel[T.FEATURES].corr(method="spearman").round(3))
print("\nturnover percentiles:",
      panel.turnover.quantile([.01,.25,.5,.75,.99]).round(4).to_dict())


## 3. The six models, both targets

Twelve model rows plus ten raw-feature references.

In [ ]:
TABLE = T.run_all(panel, CFG)


## 4. The headline table

Sorted by the size of the return spread, which is the only column comparable across targets.

In [ ]:
view = (TABLE.assign(abs_spread=TABLE.Q5_Q1_pct.abs())
             .sort_values(["target", "abs_spread"], ascending=[True, False])
             .drop(columns="abs_spread"))
for tgt in T.TARGETS:
    print(f"\n=== {tgt} ===")
    display(view[view.target == tgt].drop(columns="target").round(4).reset_index(drop=True))


In [ ]:
# model vs raw, feature by feature: a large gap means the tree added noise, not signal
cmp = []
for tgt in T.TARGETS:
    for f in T.FEATURES:
        m = TABLE[(TABLE.target == tgt) & (TABLE.model == f"model:{f}")].iloc[0]
        r = TABLE[(TABLE.target == tgt) & (TABLE.model == f"raw:{f}")].iloc[0]
        cmp.append({"target": tgt, "feature": f,
                    "IC_model": round(m.rank_IC, 4), "IC_raw": round(r.rank_IC, 4),
                    "|spread| model": round(abs(m.Q5_Q1_pct), 3),
                    "|spread| raw": round(abs(r.Q5_Q1_pct), 3)})
display(pd.DataFrame(cmp))
print("A tree on ONE feature is close to a monotone transform of it, so |IC| should match.")
print("The sign can flip -- the model learns the direction, the raw sort does not.")


## 5. Reading it

- **`ret_next`, `model:ALL` vs the best univariate** — if ALL does not beat the best single
  feature, there is no interaction worth modelling and the honest description is a
  one-characteristic sort.
- **rank_IC on `ret_next` above ~0.05** — treat as suspicious before treating as good. Returns
  are close to unforecastable; check the units line and the feature coverage first.
- **`turnover_next` predicted well, `ret_next` not** — turnover is highly persistent, so a high
  IC there is mostly autocorrelation. It says nothing about returns and should not be reported
  as if it did.
- **spread large but `spread_t` small** — a few quarters are carrying it. Check `n_quarters`.

Returns are gross: no costs, and a turnover sort concentrates in exactly the names where
trading is cheapest or dearest, so costs will not be neutral across quintiles.

In [ ]:
os.makedirs("outputs_turnover", exist_ok=True)
TABLE.to_csv("outputs_turnover/turnover_study_table.csv", index=False)
panel.to_parquet("outputs_turnover/turnover_panel.parquet", index=False)
print("saved to outputs_turnover/")
